In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from ECOv002_calval_tables import load_calval_table
from FLiESANN import load_ECOv002_calval_FLiESANN_outputs
from BESS_JPL import load_ECOv002_static_tower_BESS_inputs
from BESS_JPL import process_BESS_table

[2025-11-21 09:21:22 INFO] SRTM working directory: ~/data/NASADEM
[2025-11-21 09:21:22 INFO] SRTM download directory: ~/data/NASADEM


In [3]:
repo_root = os.path.dirname(os.getcwd())
package_dir = os.path.join(repo_root, 'BESS_JPL')
generated_input_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-inputs.csv")
generated_output_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-outputs.csv")

In [4]:
# model_inputs_gdf = load_calval_table()
model_inputs_gdf = load_ECOv002_calval_FLiESANN_outputs()
model_inputs_gdf["elevation_km"] = model_inputs_gdf["Elev"] / 1000.0
model_inputs_gdf.head()

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,NIR_diffuse_fraction,SWin_TOA_Wm2,UV_Wm2,PAR_Wm2,NIR_Wm2,PAR_diffuse_Wm2,NIR_diffuse_Wm2,PAR_direct_Wm2,NIR_direct_Wm2,elevation_km
0,0,US-NC3,ENF,Cfa,270.34520,78.53355,392.85184,307.02197,487.383423,118.91628,...,0.126794,884.3206,37.197750,297.09644,301.62590,98.470250,38.244500,198.62619,263.38138,0.005
1,1,US-Mi3,CVM,Dfb,232.14160,229.20093,640.11847,375.08930,106.825577,167.91946,...,0.000000,1199.2876,61.882423,448.33093,504.05743,34.749706,0.000000,413.58124,504.05743,0.270
2,2,US-Mi3,CVM,Dfb,356.35574,335.23154,625.66170,284.68625,NaN,132.93634,...,0.020261,1205.0444,59.888542,445.19970,478.15155,60.458492,9.687667,384.74120,468.46390,0.270
3,3,US-Mi3,CVM,Dfb,332.93840,326.68680,624.25433,251.41449,178.827545,141.13242,...,0.004152,1142.2208,57.067180,418.59454,464.95883,34.440212,1.930551,384.15433,463.02830,0.270
4,4,US-Mi3,CVM,Dfb,286.85403,237.21654,511.08218,228.52017,154.791626,114.80941,...,0.036793,1042.2742,49.484960,368.71317,398.33896,46.295307,14.656249,322.41785,383.68270,0.270


In [5]:
static_inputs_df = load_ECOv002_static_tower_BESS_inputs()
static_inputs_df

,ID,name,NDVI_minimum,NDVI_maximum,C4_fraction,carbon_uptake_efficiency,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,...,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,wind_speed_mps,vapor_gccm,ozone_cm,geometry
0,US-NC3,NC_Clearcut#3,0.408733,0.855693,0.077532,0.080000,0.410000,87.345433,51.040624,9.5,...,3,0.282353,20.642902,0,0,400,0,0,0.3,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,0.657359,0.826605,0.000549,0.060347,0.125033,41.364487,41.364487,9.5,...,1,0.254902,22.140021,0,0,400,0,0,0.3,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,0.027910,0.855461,0.035045,0.080000,0.410000,119.435443,119.435443,7.5,...,4,0.286275,0.000000,0,0,400,0,0,0.3,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,0.591358,0.869758,0.046219,0.080000,0.410000,64.720165,64.720165,9.5,...,3,0.207843,14.164827,0,0,400,0,0,0.3,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,0.399712,0.675731,0.000356,0.080000,0.410000,109.986395,109.986395,9.5,...,3,0.266667,9.919029,0,0,400,0,0,0.3,POINT (-122.9951 49.119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",-0.025449,0.495206,0.340776,0.090000,0.710000,78.000000,40.000000,9.5,...,2,0.298039,0.000000,0,0,400,0,0,0.3,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),-0.041785,0.753863,0.032479,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.294118,0.000000,0,0,400,0,0,0.3,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,-0.003026,0.776740,0.092454,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.278431,0.000000,0,0,400,0,0,0.3,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),0.233503,0.554538,0.371127,0.090000,0.710000,78.000000,40.000000,9.5,...,3,0.301961,0.000000,0,0,400,0,0,0.3,POINT (-99.0588 35.4106)


In [6]:
# merge static inputs with model inputs, ignoring duplicate columns from static_inputs_df
cols_to_use = [col for col in static_inputs_df.columns if col not in model_inputs_gdf.columns or col == 'ID']
model_inputs_gdf = model_inputs_gdf.merge(static_inputs_df[cols_to_use], on="ID", how="left")
model_inputs_gdf

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,ball_berry_slope_C4,ball_berry_intercept_C3,CI,canopy_height_meters,Ca,wind_speed_mps
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.41,87.345433,51.040624,9.5,4.951963,0.005267,0.282353,20.642902,400,0
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000,400,0
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000,400,0
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000,400,0
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000,400,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000,400,0
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000,400,0
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000,400,0
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000,400,0


In [7]:
model_inputs_gdf.columns

Index(['Unnamed: 0', 'ID', 'vegetation', 'climate', 'STICinst', 'BESSinst',
       'MOD16inst', 'PTJPLSMinst', 'ETinst', 'ETinstUncertainty',
       ...
       'kn', 'peakVCmax_C3', 'peakVCmax_C4', 'ball_berry_slope_C3',
       'ball_berry_slope_C4', 'ball_berry_intercept_C3', 'CI',
       'canopy_height_meters', 'Ca', 'wind_speed_mps'],
      dtype='object', length=124)

In [8]:
results = process_BESS_table(model_inputs_gdf)
results

[2025-11-21 09:21:22 INFO] started extracting geometry from PT-JPL-SM input table
[2025-11-21 09:21:22 INFO] completed extracting geometry from PT-JPL-SM input table
[2025-11-21 09:21:22 INFO] started extracting time from PT-JPL-SM input table
[2025-11-21 09:21:22 INFO] completed extracting time from PT-JPL-SM input table
[2025-11-21 09:21:22 INFO] variable elevation_m min: 1.000 mean: 992.883 max: 3504.000 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable Ta_C min: -14.605 mean: 22.322 max: 39.710 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable RH min: 0.273 mean: 0.427 max: 0.984 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable NDVI_minimum min: -0.033 mean: 0.174 max: 0.591 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable NDVI_maximum min: 0.314 mean: 0.628 max: 0.918 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable C4_fraction min: 0.000 mean: 0.296 max: 0.939 nan: 0.00% (nan)
[2025-11-21 09:21:22 INFO] variable carbon_uptake_efficiency min: 0.080 mean: 0.083 ma

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,wind_speed_mps,GPP,GPP_daily,Rn_Wm2,Rn_soil_Wm2,Rn_canopy_Wm2,LE_Wm2,LE_soil_Wm2,LE_canopy_Wm2,G_Wm2
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0,15.616766,5.711566,466.357577,297.381272,168.976305,226.980840,59.270467,167.710372,35.791947
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0,21.021739,8.598512,709.085679,543.104519,165.981159,217.594847,95.668201,121.926646,74.268917
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0,19.975842,8.116796,725.946606,543.887475,182.059132,320.113905,191.189198,128.924708,74.709546
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0,24.097524,10.306391,694.866890,512.976422,181.890468,320.744921,195.970016,124.774905,68.422568
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0,21.499935,10.067692,547.270243,407.281997,139.988246,240.925989,128.088996,112.836993,55.898896
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0,0.492825,0.193133,231.107920,163.076093,68.031827,45.934286,40.764830,5.169456,29.014390
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0,1.258465,0.881543,247.155614,177.971270,69.184344,40.497186,28.962675,11.534511,31.763235
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0,3.354856,2.400564,386.798873,254.162932,132.635940,46.527408,5.927288,40.600120,34.809273
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0,1.985566,1.197199,295.591819,234.818866,60.772954,56.054382,46.078914,9.975468,43.789133


In [9]:
model_inputs_gdf.to_csv(generated_input_table_filename, index=False)

In [10]:
results.to_csv(generated_output_table_filename, index=False)

In [11]:
for key, value in results.items():
    try:
        print(f"{key}: {np.nanmin(value)} {np.nanmean(value)} {np.nanmax(value)}")
    except:
        continue

Unnamed: 0: 0 532.0 1064
STICinst: 0.0 163.1636612074366 523.0476
BESSinst: 0.0 213.8509916843192 1000.0
MOD16inst: 0.0 294.62475544413144 762.08057
PTJPLSMinst: 0.0 171.57670828262908 526.5678
ETinst: 0.0 162.73812873411688 741.0538940429688
ETinstUncertainty: 0.0 123.6068336347418 420.74088
PET: 0.0 297.61518383230043 695.64197
Rn: 0.0 414.2791522507042 763.1359
ESI: 0.0 0.5617755694967136 1.0
RH: 0.27253073 0.42692000418779347 0.98366296
Ta: -14.605048 22.321587967441317 39.710495
LST: 258.72 302.1800469295775 359.26
SM: 0.0 0.16790158060469482 0.89711
NDVI: -0.02429185 0.4528892517239436 0.94546
NDVI-UQ: 0.00053067924 0.009029797618441315 0.09791492
albedo: 0.015407953 0.10903688726384977 0.6173986
albedo-UQ: 0.00057431095 0.006081434030807512 0.0493286
LST_err: 0.52 1.1144413145539904 2.96
view_zenith: 0.5024103 15.425197045624413 29.96069
Rg: -23.763361 606.9121518676056 1042.9371
EmisWB: 0.734 0.9682704225821597 0.984
solar_hour: 6.0 11.597183098591549 17.0
LE: -28.40977018 106.